### Лучшее решение на 0.86

In [ ]:
import os
import re
import warnings
import time

import numpy as np
import pandas as pd
from scipy.sparse import hstack as sparse_hstack

import nltk
from nltk.tokenize import word_tokenize
from nltk import pos_tag

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.neural_network import MLPClassifier
from sklearn.base import clone

try:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_HGB = True
except ImportError:
    HAS_HGB = False

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

from gensim.models import FastText, Phrases

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)
try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger', quiet=True)

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\.{3,}', ' ELLIPSIS ', text)
    text = re.sub(r'!+', ' EXMARK ', text)
    text = re.sub(r'\?+', ' QMARK ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    try:
        tokens = word_tokenize(text)
    except Exception:
        tokens = text.split()
    tokens = [t for t in tokens if len(t) > 1 or t in {'i', 'a'}]
    return ' '.join(tokens)

In [ ]:
LEXICONS = {
    'first_person': {'i', 'me', 'my', 'myself', 'mine'},
    'negation': {"n't", 'not', 'no', 'never', 'nothing', 'nobody', 'nowhere', 'neither', 'none', 'hardly', 'barely', 'scarcely', 'without'},
    'absolutist': {'always', 'never', 'completely', 'totally', 'absolutely', 'entirely', 'constantly', 'all', 'everyone', 'everybody', 'everything', 'everywhere', 'forever', 'definitely', 'certainly', 'utterly', 'nothing', 'none'},
    'sadness': {'sad', 'depressed', 'depression', 'anxiety', 'anxious', 'hopeless', 'worthless', 'empty', 'lonely', 'alone', 'crying', 'cry', 'tears', 'pain', 'hurt', 'suffering', 'miserable', 'terrible', 'awful', 'hate', 'hated', 'suicide', 'suicidal', 'die', 'dead', 'death', 'kill', 'killing', 'broken', 'lost', 'dark', 'numb', 'tired', 'exhausted', 'failure', 'useless', 'burden', 'regret', 'guilty', 'shame', 'scared', 'fear', 'worried', 'stress', 'stressed', 'overwhelmed', 'trapped', 'stuck', 'isolated', 'rejected', 'abandoned', 'helpless', 'despair', 'meaningless', 'pointless', 'void', 'disappoint', 'disappointed', 'disappointing'},
    'positive': {'happy', 'joy', 'love', 'loved', 'loving', 'great', 'good', 'amazing', 'wonderful', 'fantastic', 'excellent', 'perfect', 'best', 'better', 'awesome', 'excited', 'glad', 'pleased', 'grateful', 'thankful', 'hope', 'hopeful', 'proud', 'confident', 'calm', 'peaceful', 'relaxed', 'cheerful', 'delighted', 'blessed', 'lucky', 'smile', 'laughing', 'fun', 'enjoy', 'enjoyed', 'enjoying', 'beautiful', 'nice', 'lovely', 'sweet', 'kind', 'success', 'successful', 'win', 'winning', 'achievement', 'accomplished'},
    'anger': {'angry', 'anger', 'mad', 'furious', 'rage', 'hate', 'hatred', 'annoyed', 'irritated', 'frustrated', 'pissed'},
    'social': {'friend', 'friends', 'family', 'mother', 'father', 'mom', 'dad', 'parent', 'parents', 'brother', 'sister', 'sibling', 'wife', 'husband', 'partner', 'boyfriend', 'girlfriend', 'son', 'daughter', 'child', 'children', 'people', 'person', 'someone', 'everyone', 'anyone', 'talk', 'talking', 'communicate', 'conversation', 'together', 'relationship'},
    'cognitive': {'think', 'thought', 'thinking', 'know', 'knew', 'known', 'believe', 'consider', 'wonder', 'realize', 'understand', 'remember', 'decide', 'decision', 'problem', 'problems', 'reason', 'because', 'cause', 'effect'},
    'insight': {'think', 'thought', 'thinking', 'know', 'knew', 'aware', 'realize', 'realized', 'understand', 'understood', 'recognize', 'recognised', 'figure', 'sense'},
    'tentative': {'maybe', 'perhaps', 'possibly', 'probably', 'guess', 'suppose', 'seems', 'appear', 'somewhat', 'sort', 'kinda', 'kindof'},
    'certainty': {'always', 'never', 'definitely', 'certainly', 'absolutely', 'completely', 'totally', 'exactly', 'precisely', 'obviously', 'clearly'},
    'inhibition': {'stop', 'stopped', 'block', 'blocked', 'constrain', 'restrain', 'control', 'controlled', 'prevent', 'avoid', 'avoided', 'quit', 'quitted'},
    'death': {'die', 'died', 'dead', 'death', 'kill', 'killed', 'killing', 'suicide', 'suicidal', 'overdose', 'funeral', 'grave', 'buried', 'afterlife'},
    'biology': {'body', 'health', 'healthy', 'sick', 'illness', 'disease', 'pain', 'hurt', 'blood', 'heart', 'brain', 'head', 'stomach', 'sleep', 'sleeping', 'eat', 'eating', 'appetite', 'weight'},
    'achievement': {'achieve', 'achievement', 'goal', 'success', 'successful', 'accomplish', 'accomplished', 'win', 'won', 'beat', 'pass', 'excel', 'effort', 'try', 'trying'},
}

In [ ]:
def count_syllables_word(word):
    word = word.lower()
    vowels = 'aeiouy'
    syl = 0
    prev_vowel = False
    for ch in word:
        is_v = ch in vowels
        if is_v and not prev_vowel:
            syl += 1
        prev_vowel = is_v
    if word.endswith('e') and syl > 1:
        syl -= 1
    return max(syl, 1)


def extract_meta(df):
    meta = pd.DataFrame(index=df.index)
    txt = df['original_text'].astype(str)
    toks = df['clean_text'].fillna('').str.lower().str.split()
    words = txt.str.lower().str.findall(r"\b\w+\b")

    title_toks = df['title'].fillna('').astype(str).str.lower().str.split()
    body_toks = df['body'].fillna('').astype(str).str.lower().str.split()

    meta['char_count'] = txt.apply(len)
    meta['word_count'] = toks.apply(len)
    meta['avg_word_len'] = toks.apply(lambda x: np.mean([len(w) for w in x]) if x else 0)
    meta['sentence_count'] = txt.apply(lambda x: max(len(re.split(r'[.!?]+', x)) - 1, 1))
    meta['avg_sent_len'] = meta['word_count'] / meta['sentence_count'].clip(lower=1)

    meta['exclamation'] = txt.apply(lambda x: x.count('!'))
    meta['question'] = txt.apply(lambda x: x.count('?'))
    meta['ellipsis'] = txt.apply(lambda x: x.count('...'))
    meta['comma'] = txt.apply(lambda x: x.count(','))
    meta['upper_ratio'] = txt.apply(lambda x: sum(1 for c in x if c.isupper()) / max(len(x), 1))
    meta['punct_ratio'] = (meta['exclamation'] + meta['question'] + meta['ellipsis'] + meta['comma']) / meta['char_count'].clip(lower=1)

    meta['ttr'] = words.apply(lambda x: len(set(x)) / max(len(x), 1))

    def readability(word_list, sent_count):
        if not word_list or sent_count == 0:
            return 0, 0, 0
        n_words = len(word_list)
        n_chars = sum(len(w) for w in word_list)
        n_syl = sum(count_syllables_word(w) for w in word_list)
        flesch = 206.835 - 1.015 * (n_words / sent_count) - 84.6 * (n_syl / n_words)
        ari = 4.71 * (n_chars / n_words) + 0.5 * (n_words / sent_count) - 21.43
        L = 100 * n_chars / n_words
        S = 100 * sent_count / n_words
        cli = 0.0588 * L - 0.296 * S - 15.8
        return flesch, ari, cli

    read = words.combine(meta['sentence_count'], lambda w, s: readability(w, s))
    meta['flesch'] = read.apply(lambda x: x[0])
    meta['ari'] = read.apply(lambda x: x[1])
    meta['cli'] = read.apply(lambda x: x[2])

    def pos_ratios(token_list):
        if not token_list:
            return {k: 0 for k in ['NOUN', 'VERB', 'ADJ', 'ADV', 'PRON', 'DET', 'ADP', 'NUM', 'CONJ', 'PRT', 'X']}
        try:
            tags = [t[1] for t in pos_tag(token_list, tagset='universal')]
        except Exception:
            tags = []
        n = max(len(tags), 1)
        return {
            'NOUN': tags.count('NOUN') / n,
            'VERB': tags.count('VERB') / n,
            'ADJ': tags.count('ADJ') / n,
            'ADV': tags.count('ADV') / n,
            'PRON': tags.count('PRON') / n,
            'DET': tags.count('DET') / n,
            'ADP': tags.count('ADP') / n,
            'NUM': tags.count('NUM') / n,
            'CONJ': tags.count('CONJ') / n,
            'PRT': tags.count('PRT') / n,
            'X': tags.count('X') / n,
        }

    pos_df = pd.DataFrame(toks.apply(pos_ratios).tolist(), index=df.index)
    for col in pos_df.columns:
        meta[f'pos_{col.lower()}'] = pos_df[col]

    wc = meta['word_count'].clip(lower=1)
    for cat, lex in LEXICONS.items():
        meta[f'{cat}_count'] = toks.apply(lambda x: sum(1 for t in x if t in lex))
        meta[f'{cat}_ratio'] = meta[f'{cat}_count'] / wc

    def lex_ratio(token_list, lexicon):
        if not token_list:
            return 0
        return sum(1 for t in token_list if t in lexicon) / len(token_list)

    meta['title_sadness'] = title_toks.apply(lambda x: lex_ratio(x, LEXICONS['sadness']))
    meta['body_sadness'] = body_toks.apply(lambda x: lex_ratio(x, LEXICONS['sadness']))
    meta['title_positive'] = title_toks.apply(lambda x: lex_ratio(x, LEXICONS['positive']))
    meta['body_positive'] = body_toks.apply(lambda x: lex_ratio(x, LEXICONS['positive']))
    meta['title_negation'] = title_toks.apply(lambda x: lex_ratio(x, LEXICONS['negation']))
    meta['body_negation'] = body_toks.apply(lambda x: lex_ratio(x, LEXICONS['negation']))
    # добавляем в лучшем решении
    meta['title_fp'] = title_toks.apply(lambda x: lex_ratio(x, LEXICONS['first_person']))
    meta['body_fp'] = body_toks.apply(lambda x: lex_ratio(x, LEXICONS['first_person']))
    # -
    meta['sentiment_ratio'] = meta['sadness_ratio'] - meta['positive_ratio']
    meta['emotion_intensity'] = meta['sadness_ratio'] + meta['anger_ratio'] + meta['positive_ratio']
    meta['self_focus'] = meta['first_person_ratio'] + meta['insight_ratio']
    # добавляем в лучшем решении
    meta['temporal_shift'] = meta['past_focus_ratio'] - meta['future_focus_ratio']
    # -
    return meta

In [ ]:
class SuperFeatureExtractor:
    def __init__(self,
                 ft_size=120, # 150
                 ft_window=8,
                 ft_min_count=2,
                 ft_epochs=20, # 25
                 word_tfidf_max=15000,
                 word_svd=300,
                 char_tfidf_max=8000,
                 char_svd=100):
        self.ft_size = ft_size
        self.ft_window = ft_window
        self.ft_min_count = ft_min_count
        self.ft_epochs = ft_epochs
        self.word_tfidf_max = word_tfidf_max
        self.word_svd = word_svd
        self.char_tfidf_max = char_tfidf_max
        self.char_svd = char_svd

        self.ft = None
        self.phraser = None
        self.word_tfidf = None
        self.idf = None
        self.word_svd = TruncatedSVD(n_components=word_svd, random_state=RANDOM_STATE)
        self.char_tfidf = None
        self.char_svd = TruncatedSVD(n_components=char_svd, random_state=RANDOM_STATE)
        self.scaler = StandardScaler()

    def fit(self, texts):
        tokenized = [t.split() for t in texts if t.split()]

        self.phraser = Phrases(tokenized, min_count=5, threshold=10)
        tok_phrased = [self.phraser[t] for t in tokenized]

        self.ft = FastText(
            sentences=tok_phrased,
            vector_size=self.ft_size,
            window=self.ft_window,
            min_count=self.ft_min_count,
            workers=4,
            sg=1,
            epochs=self.ft_epochs,
            seed=RANDOM_STATE,
            word_ngrams=1,
            # min_n = 3, max_n = 6
        )

        self.word_tfidf = TfidfVectorizer(
            max_features=self.word_tfidf_max,
            ngram_range=(1, 3),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
        word_mat = self.word_tfidf.fit_transform(texts)
        self.idf = dict(zip(self.word_tfidf.get_feature_names_out(), self.word_tfidf.idf_))
        self.word_svd.fit(word_mat)

        self.char_tfidf = TfidfVectorizer(
            analyzer='char',
            ngram_range=(2, 4),
            max_features=self.char_tfidf_max,
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
        char_mat = self.char_tfidf.fit_transform(texts)
        self.char_svd.fit(char_mat)
        return self

    def _phrase(self, text):
        return self.phraser[text.split()] if self.phraser else text.split()

    def _ft_mean(self, text):
        toks = self._phrase(text)
        vecs = [self.ft.wv[t] for t in toks if t in self.ft.wv]
        return np.mean(vecs, axis=0) if vecs else np.zeros(self.ft_size)

    def _ft_std(self, text):
        toks = self._phrase(text)
        vecs = [self.ft.wv[t] for t in toks if t in self.ft.wv]
        return np.std(vecs, axis=0) if vecs else np.zeros(self.ft_size)

    def _ft_tfidf(self, text):
        toks = self._phrase(text)
        vecs, weights = [], []
        for t in toks:
            if t in self.ft.wv and t in self.idf:
                w = self.idf[t]
                vecs.append(self.ft.wv[t] * w)
                weights.append(w)
        if not vecs:
            return np.zeros(self.ft_size)
        return np.sum(vecs, axis=0) / (np.sum(weights) + 1e-9)

    def transform(self, texts, meta_df):
        ft_mean = np.vstack([self._ft_mean(t) for t in texts])
        ft_std = np.vstack([self._ft_std(t) for t in texts])
        ft_tfidf = np.vstack([self._ft_tfidf(t) for t in texts])
        w_svd = self.word_svd.transform(self.word_tfidf.transform(texts))
        c_svd = self.char_svd.transform(self.char_tfidf.transform(texts))
        meta = self.scaler.transform(meta_df)

        dense = np.hstack([ft_mean, ft_std, ft_tfidf, w_svd, c_svd, meta])
        sparse = sparse_hstack([self.word_tfidf.transform(texts), self.char_tfidf.transform(texts)], format='csr')
        return dense, sparse

    def fit_transform(self, texts, meta_df):
        self.fit(texts)
        self.scaler.fit(meta_df)
        return self.transform(texts, meta_df)

In [ ]:
class HybridStacking:
    def __init__(self, sparse_models, dense_models, meta_model, n_splits=5):
        self.sparse_models = sparse_models
        self.dense_models = dense_models
        self.meta_model = meta_model
        self.n_splits = n_splits
        self.sparse_trained_ = []
        self.dense_trained_ = []
        self.meta_trained_ = None
        self.oof_probs_ = None

    def fit(self, X_dense, X_sparse, y):
        skf = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=RANDOM_STATE)
        n_sparse = len(self.sparse_models)
        n_dense = len(self.dense_models)
        oof = np.zeros((len(y), n_sparse + n_dense))

        for fold, (tr_i, val_i) in enumerate(skf.split(X_dense, y), 1):
            print(f"  [Hybrid] Fold {fold}/{self.n_splits}")
            t0 = time.time()

            for m_idx, (name, model) in enumerate(self.sparse_models):
                m = clone(model)
                m.fit(X_sparse[tr_i], y[tr_i])
                oof[val_i, m_idx] = m.predict_proba(X_sparse[val_i])[:, 1]

            for m_idx, (name, model) in enumerate(self.dense_models):
                m = clone(model)
                m.fit(X_dense[tr_i], y[tr_i])
                oof[val_i, n_sparse + m_idx] = m.predict_proba(X_dense[val_i])[:, 1]

            print(f"      Fold done in {time.time()-t0:.1f}s")

        self.oof_probs_ = oof

        self.meta_trained_ = clone(self.meta_model)
        self.meta_trained_.fit(oof, y)

        for name, model in self.sparse_models:
            m = clone(model)
            m.fit(X_sparse, y)
            self.sparse_trained_.append(m)
        for name, model in self.dense_models:
            m = clone(model)
            m.fit(X_dense, y)
            self.dense_trained_.append(m)
        return self

    def predict_proba(self, X_dense, X_sparse):
        n_sparse = len(self.sparse_trained_)
        n_dense = len(self.dense_trained_)
        base = np.zeros((len(X_dense), n_sparse + n_dense))

        for m_idx, model in enumerate(self.sparse_trained_):
            base[:, m_idx] = model.predict_proba(X_sparse)[:, 1]
        for m_idx, model in enumerate(self.dense_trained_):
            base[:, n_sparse + m_idx] = model.predict_proba(X_dense)[:, 1]

        return self.meta_trained_.predict_proba(base)[:, 1]

    def oof_proba(self):
        return self.meta_trained_.predict_proba(self.oof_probs_)[:, 1]


def best_threshold(y_true, y_prob):
    best_f1, best_t = 0.0, 0.5
    for t in np.arange(0.05, 0.96, 0.005): # 0.001
        f1 = f1_score(y_true, (y_prob >= t).astype(int))
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier


def run_super(train_path, test_path, out_path='submission.csv'):

    tr = pd.read_csv(train_path)
    te = pd.read_csv(test_path)
    print(f"      Train {tr.shape} | Test {te.shape}")

    for df in (tr, te):
        df['title'] = df['title'].fillna('')
        df['body'] = df['body'].fillna('')
        df['original_text'] = (df['title'] + ' ' + df['body']).astype(str)
        df['clean_text'] = df['original_text'].apply(clean_text)

    tr_meta = extract_meta(tr)
    te_meta = extract_meta(te)

    fe = SuperFeatureExtractor(
        ft_size=120, # 150
        ft_window=8,
        ft_min_count=2,
        ft_epochs=20, # 25
        word_tfidf_max=15000,
        word_svd=300,
        char_tfidf_max=8000,
        char_svd=100
    )
    X_dense, X_sparse = fe.fit_transform(tr['clean_text'].values, tr_meta)
    X_te_dense, X_te_sparse = fe.transform(te['clean_text'].values, te_meta)
    y = tr['label'].values.astype(int)
    print(f"      Dense: {X_dense.shape} | Sparse: {X_sparse.shape}")


    sparse_models = [
        ('lr1', LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0,
                                   solver='saga', random_state=RANDOM_STATE)),
        ('lr2', LogisticRegression(max_iter=1000, class_weight='balanced', C=0.05, #0.03
                                   solver='saga', random_state=RANDOM_STATE)),
        ('svc', CalibratedClassifierCV(
            LinearSVC(C=0.3, class_weight='balanced', max_iter=3000, random_state=RANDOM_STATE),
            method='sigmoid', cv=3
        )),
    ]

    dense_models = []

    if HAS_HGB:
        dense_models.append(('hgb', HistGradientBoostingClassifier(
            # 250, 0.06
            max_iter=200, learning_rate=0.08, max_depth=5,
            min_samples_leaf=20, early_stopping=True,
            validation_fraction=0.1, n_iter_no_change=10,
            random_state=RANDOM_STATE
        )))
    else:
        dense_models.append(('gb', GradientBoostingClassifier(
            # 100
            n_estimators=80, max_depth=3, learning_rate=0.1,
            subsample=0.8, random_state=RANDOM_STATE
        )))

    if HAS_XGB:
        neg, pos = np.bincount(y)
        spw = neg / pos
        dense_models.append(('xgb', XGBClassifier(
            # 400, 0.6
            n_estimators=300, max_depth=4, learning_rate=0.08,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=spw,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            n_jobs=2
        )))
    else:
        dense_models.append(('rf', RandomForestClassifier(
            # 20
            n_estimators=200, max_depth=15, min_samples_split=5,
            class_weight='balanced', n_jobs=2, random_state=RANDOM_STATE
        )))

    dense_models.append(('mlp', MLPClassifier(
        hidden_layer_sizes=(256, 128),
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        max_iter=500,
        random_state=RANDOM_STATE
    )))

    meta = XGBClassifier(
        #150
        n_estimators=100, max_depth=3, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=2
    )

    ensemble = HybridStacking(sparse_models, dense_models, meta, n_splits=5)
    ensemble.fit(X_dense, X_sparse, y)

    oof_prob = ensemble.oof_proba()
    thresh, f1 = best_threshold(y, oof_prob)
    print(f"Best threshold: {thresh:.3f} | OOF F1: {f1:.4f}")
    print("\n      OOF Classification Report:")
    print(classification_report(y, (oof_prob >= thresh).astype(int),
                                target_names=['Not Depr', 'Depr'], digits=4))

    prob_te = ensemble.predict_proba(X_te_dense, X_te_sparse)
    pred_te = (prob_te >= thresh).astype(int)

    sub = pd.DataFrame({'id': te['id'], 'label': pred_te})
    sub.to_csv(out_path, index=False)
    print(f" SAVED: {out_path}")
    print(sub['label'].value_counts())        

In [ ]:
train = 'train.csv'
test = 'test.csv'
out = 'submission.csv'
run_super(train, test, out)